# Recommendation DataFrame Inspector
This notebook allows for the inspection and comparison of `recommendation_df` outputs generated by the `RecommendationEngine` without recomputing the pipeline. It retrieves data directly from the parquet cache.

## 1. Environment Setup
We establish the project root and update `sys.path` to ensure we can import the `vqs` submodule and the `configs` folder.

In [2]:
import sys
import os
from pathlib import Path
import pandas as pd

# Start from the current working directory
cwd = Path.cwd()
PROJECT_ROOT: Path | None = None

# Search upwards (max 5 levels) for the project root
for _ in range(5):
    if (cwd / "dependencies").is_dir() and (cwd / "vqs").is_dir():
        PROJECT_ROOT = cwd
        break
    cwd = cwd.parent

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find project root. Run Jupyter from within your project folder.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)
print(f"Project Root: {PROJECT_ROOT}")
print(f"CWD set to: {os.getcwd()}")

Project Root: c:\Users\liv\git\thesis\vaa-question-similarity
CWD set to: c:\Users\liv\git\thesis\vaa-question-similarity


## 2. Load Configuration and Cache
We load the `base_constants.py` file. 

**Note:** If you want to inspect a specific run, ensure the parameters in the cell below match those used during that run.

In [12]:
from main import load_config
from vqs.cache_management import CacheManager

# Load default config
config_path = PROJECT_ROOT / "configs" / "full_pipeline" / "pipeline_e5_ZH_mini.py" #type: ignore
config = load_config(config_path)

# Replicate RecommendationEngine parameter list for hashing
important_params_list = [
    "data_year", "dist", "alpha", "crw_paper_choice", "rec_dist_method",
    "n_recommendations", "subset_n", "filter_districts"
]
if config.filter_districts:
    important_params_list.append("district")
if getattr(config, 'E5_instruction', None) is not None:
    important_params_list.append("E5_instruction")

# Construct prefix exactly as evaluate_pipeline does
prefix = f"recs_{config.data_year}_{config.dist}_a{config.alpha}_subset={config.subset_n}"
if config.filter_districts:
    prefix += f"_{config.district}"

cacher = CacheManager(
    config=config,
    cache_dir=config.RECOMMENDATION_CACHE_DIR,
    prefix=prefix,
    params_list=important_params_list
)

recommendation_df = cacher.load_if_exists()

if recommendation_df is not None:
    print("✅ Cache successfully loaded.")
else:
    print("❌ No cache found. Check your parameters or prefix logic.")

--- Cache Hit: recs_2023_E5_a0.6_subset=1000_ZH_5de9297c87d2f1262c18f5ecde28b9c4.parquet ---
✅ Cache successfully loaded.


## 3. Inspection of Headers and Structure
Here we look at the column organization. The dataframe should contain both baseline columns and CRW-prefixed columns.

In [13]:
if recommendation_df is not None:
    print(f"DataFrame Shape: {recommendation_df.shape}")

    rec_cols = [c for c in recommendation_df.columns]

    print(f"\n--- Recommendation Columns ({len(rec_cols)}) ---")
    for col in rec_cols:
        print(col)


DataFrame Shape: (1000, 343)

--- Recommendation Columns (343) ---
recID
user_language
voterID
researchID
electionID
districtID
sourceTYPE
source
recTYPE
questTYPE
language
birthYEAR
gender
zip
education
interest
position
pref_party
soc_completion
N_answers
quest_completion
answer_32214
answer_32215
answer_32216
answer_32217
answer_32218
answer_32219
answer_32220
answer_32221
answer_32222
answer_32223
answer_32224
answer_32225
answer_32226
answer_32227
answer_32228
answer_32229
answer_32230
answer_32231
answer_32232
answer_32233
answer_32234
answer_32235
answer_32236
answer_32237
answer_32238
answer_32239
answer_32240
answer_32241
answer_32242
answer_32243
answer_32244
answer_32245
answer_32246
answer_32247
answer_32248
answer_32249
answer_32250
answer_32251
answer_32252
answer_32253
answer_32254
answer_32255
answer_32256
answer_32257
answer_32258
answer_32259
answer_32260
answer_32261
answer_32262
answer_32263
answer_32264
answer_32265
answer_32266
answer_32267
answer_32268
answer_322

In [20]:
if recommendation_df is not None:
    x: int = 1 # 1 <= x <= n_recommendations
    print(f"---- Top {x} match IDs (first 20 rows) ---")
    print(recommendation_df[f"_matchID_{x}_L2_sv"].iloc[:20])
    print(recommendation_df[f"CRW__matchID_{x}_L2_sv"].iloc[:20])

---- Top 1 match IDs (first 20 rows) ---
0     58681.0
6     54684.0
7     57045.0
9     57763.0
21    58559.0
23    58882.0
24    57394.0
34    54513.0
41    57164.0
42    58919.0
43    58008.0
44    55079.0
57    58497.0
61    57931.0
66    58846.0
67    57381.0
69    58497.0
76    58008.0
82    57786.0
84    54515.0
Name: _matchID_1_L2_sv, dtype: float64
0     58681.0
6     57301.0
7     57045.0
9     57893.0
21    57924.0
23    58882.0
24    57394.0
34    54513.0
41    57164.0
42    58919.0
43    58008.0
44    55079.0
57    58497.0
61    57931.0
66    58846.0
67    57381.0
69    57163.0
76    56919.0
82    57786.0
84    54515.0
Name: CRW__matchID_1_L2_sv, dtype: float64


## 4. Compare Different Outputs (Extension)
To compare different engine runs (e.g., different Alpha values), you can manually point to specific files in the cache directory.

In [ ]:
import glob

# List all cached recommendation files to see what's available
cache_files = glob.glob(str(config.RECOMMENDATION_CACHE_DIR / "*.parquet"))
print("Available cache files:")
for f in cache_files:
    print(Path(f).name)

def load_specific_cache(filename):
    path = config.RECOMMENDATION_CACHE_DIR / filename
    return pd.read_parquet(path)

# Example: Compare top 1 matches between two runs
# df_v1 = load_specific_cache("some_file_a0.6.parquet")
# df_v2 = load_specific_cache("some_file_a0.8.parquet")